In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/incredet/HyperSpectralSuperResolution.git 2>/dev/null || echo 'Repo already cloned'
!pip install -q -r /content/HyperSpectralSuperResolution/requirements.txt

In [ ]:
# Update SAMPLES and CKPT to where you placed the sample data and pre-trained
# checkpoint linked in the README.
SAMPLES = '/content/drive/MyDrive/sr_demo/zips_cnmf_npy'
CKPT    = '/content/drive/MyDrive/sr_demo/exp1-rrdb-192x24-cnmf-best.pth'
CONFIG  = 'exp1_rrdb_192x24'

In [ ]:
%cd /content/HyperSpectralSuperResolution/training
import yaml, torch
from pathlib import Path
from model import build_model, load_checkpoint

cfg = yaml.safe_load(Path(f'configs/{CONFIG}.yaml').read_text())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = build_model(cfg, device)
kind = load_checkpoint(net, CKPT)
net.eval()
print(f'Loaded {kind} weights from {CKPT}')
print(f'Model: {cfg.get("model_type", "rrdbnet6x")} on {device}')

In [ ]:
import numpy as np
import torch.nn.functional as F
from dataset import build_index, read_tif_from_zip, EMIT_SCALE, EMIT_NODATA

index = build_index(SAMPLES, cfg['gt_source'])
if not index:
    raise SystemExit(f'No tiles found in {SAMPLES} for gt_source={cfg["gt_source"]}')

zip_path, lr_name, gt_name, scene_id = index[0]
lq = read_tif_from_zip(zip_path, lr_name)
gt = read_tif_from_zip(zip_path, gt_name)

mask = lq == EMIT_NODATA
lq_norm = lq.astype(np.float32) / EMIT_SCALE
lq_norm[mask] = 0.0
gt_norm = np.clip(gt.astype(np.float32) / EMIT_SCALE, 0, 1.5)

with torch.no_grad():
    sr = net(torch.from_numpy(lq_norm[None]).to(device)).squeeze(0).cpu().numpy()
bic = F.interpolate(
    torch.from_numpy(lq_norm[None]).float(),
    scale_factor=cfg['scale'], mode='bicubic', align_corners=False,
).squeeze(0).numpy()

print(f'Tile: {scene_id} / {lr_name}')
print(f'LR: {lq.shape}  GT: {gt.shape}  SR: {sr.shape}')

In [ ]:
from viz import compute_all_metrics, make_main_figure
import matplotlib.pyplot as plt

border = cfg['scale']
m = compute_all_metrics(sr, gt_norm, bic, cfg['scale'], border)
print(f"SR:      PSNR={m['sr_psnr']:.2f}  SAM={m['sr_sam']:.2f}  ERGAS={m['sr_ergas']:.2f}")
print(f"Bicubic: PSNR={m['bic_psnr']:.2f}  SAM={m['bic_sam']:.2f}  ERGAS={m['bic_ergas']:.2f}")

vis_bands = cfg.get('vis_bands', [10, 6, 2])
fig = make_main_figure(sr, gt_norm, bic, m, vis_bands, cfg['gt_source'])
plt.show()